In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42
PLOT_DPI = 600
TOP_N_SHAP = 10

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = PLOT_DPI
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_figure(fig, save_path, dpi=PLOT_DPI):
    fig.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )
    plt.close(fig)


def save_confusion_matrix(cm, class_names, save_path, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )

    thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_roc_curve(y_true, y_prob, save_path):
    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Test ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_pr_curve(y_true, y_prob, save_path):
    ap = average_precision_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(recall, precision, linewidth=2, label=f"AP = {ap:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Test Precision-Recall Curve")
    ax.legend(loc="lower left")
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_test_score_distribution(y_true, y_prob, save_path):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.hist(
        y_prob[y_true == 0],
        bins=30,
        alpha=0.7,
        label="LTS (0)",
        edgecolor="black",
    )
    ax.hist(
        y_prob[y_true == 1],
        bins=30,
        alpha=0.7,
        label="HTS (1)",
        edgecolor="black",
    )
    ax.set_xlabel("Predicted probability of HTS")
    ax.set_ylabel("Cell count")
    ax.set_title("Test Set Prediction Score Distribution")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_hts_associated_shap_bar(shap_importance_df, top_n, save_path):
    """Plot HTS-associated genes using coefficient direction and SHAP importance.

    A positive logistic-regression coefficient indicates that higher expression
    shifts the prediction toward the positive class (HTS). Mean absolute SHAP
    values are used to rank the overall importance of those genes.
    """
    hts_df = shap_importance_df[
        shap_importance_df["coefficient"] > 0
    ].copy()
    hts_df = hts_df.sort_values("mean_abs_shap", ascending=False).head(top_n)

    if hts_df.empty:
        print("[WARNING] 没有 coefficient > 0 的基因，无法绘制 HTS-associated 条形图。")
        return

    hts_df = hts_df.iloc[::-1]

    fig, ax = plt.subplots(figsize=(9, max(6, hts_df.shape[0] * 0.45)))
    ax.barh(
        hts_df["gene"],
        hts_df["mean_abs_shap"],
        color="#d95f02",
        edgecolor="black",
        linewidth=0.7,
    )
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(
        f"Top {hts_df.shape[0]} HTS-associated Genes by SHAP Importance"
    )
    fig.tight_layout()
    save_figure(fig, save_path)


def plot_topn_shap_importance(shap_df, save_path, top_n=TOP_N_SHAP):
    df = (
        shap_df.sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .copy()
        .iloc[::-1]
    )

    fig, ax = plt.subplots(figsize=(8.5, max(6, top_n * 0.45)))
    sc = ax.scatter(
        df["mean_abs_shap"],
        df["gene"],
        c=df["mean_abs_shap"],
        cmap="RdYlBu_r",
        s=100,
        edgecolors="black",
        linewidths=0.3,
    )

    ax.set_xlabel("Mean |SHAP value|")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {top_n} Gene Importance by SHAP")
    fig.colorbar(sc, ax=ax, label="Importance")
    fig.tight_layout()
    save_figure(fig, save_path)


def save_shap_summary_plot(shap_values, X_scaled, feature_cols, save_path):
    import shap

    try:
        custom_cmap = LinearSegmentedColormap.from_list(
            "custom_shap_cmap",
            ["#39489f", "#39bbec", "#f9ed36", "#f38466", "#b81f25"],
            N=256,
        )

        shap.summary_plot(
            shap_values,
            X_scaled,
            feature_names=feature_cols,
            show=False,
            cmap=custom_cmap,
        )

        fig = plt.gcf()
        fig.set_size_inches(8, 8)
        save_figure(fig, save_path)
        print(f"[INFO] SHAP summary plot saved: {save_path}")
    except Exception as exc:
        print(f"[WARNING] SHAP summary 图绘制失败：{exc}")


def build_pipeline(C, solver, max_iter, class_weight):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "lr",
                LogisticRegression(
                    C=C,
                    solver=solver,
                    max_iter=max_iter,
                    class_weight=class_weight,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def get_linear_shap_values(model, background, data):
    """Return a two-dimensional SHAP-value array across SHAP versions."""
    import shap

    explainer = shap.LinearExplainer(model, background)

    try:
        explanation = explainer(data)
        shap_values = np.asarray(explanation.values)
    except Exception:
        shap_values = np.asarray(explainer.shap_values(data))

    if shap_values.ndim == 3 and shap_values.shape[-1] == 2:
        shap_values = shap_values[:, :, 1]
    elif shap_values.ndim == 3 and shap_values.shape[0] == 2:
        shap_values = shap_values[1]

    if shap_values.ndim != 2:
        raise ValueError(f"Unexpected SHAP shape: {shap_values.shape}")

    return shap_values


def main():
    code_dir = Path(".").resolve()
    data_path = code_dir / "df_expr.csv"
    out_dir = code_dir / "lr_hts_lts_results"
    fig_dir = out_dir / "figures"
    ensure_dir(out_dir)
    ensure_dir(fig_dir)

    print(f"[INFO] Working directory: {code_dir}")
    print(f"[INFO] Reading file: {data_path}")

    if not data_path.exists():
        raise FileNotFoundError(f"找不到文件: {data_path}")

    df = pd.read_csv(data_path)

    if "label" not in df.columns:
        raise ValueError("数据中必须包含 'label' 列。")

    non_feature_cols = ["label"]
    if df.columns[0] != "label":
        first_col = df.columns[0]
        if not pd.api.types.is_numeric_dtype(df[first_col]):
            non_feature_cols.append(first_col)

    feature_cols = [c for c in df.columns if c not in non_feature_cols]

    if not feature_cols:
        raise ValueError("没有可用特征列，请检查 df_expr.csv 格式。")

    X = df[feature_cols].copy()
    y = df["label"].copy()

    # Binary labels: LTS = 0; HTS = 1.
    y = y.map({0: 0, 1: 1})
    if y.isna().any():
        raise ValueError("label 列只能包含 0 和 1，其中 0=LTS、1=HTS。")

    X_numeric = X.apply(pd.to_numeric, errors="coerce")
    n_missing = int(X_numeric.isna().sum().sum())
    if n_missing > 0:
        print(
            f"[WARNING] 特征矩阵中有 {n_missing} 个缺失或无法转换的值，"
            "将其填充为 0。"
        )
    X = X_numeric.fillna(0.0).astype(np.float32)
    y = y.astype(int)

    print(f"[INFO] Data shape: {df.shape}")
    print(f"[INFO] Feature matrix shape: {X.shape}")
    print(f"[INFO] HTS (1): {(y == 1).sum()}, LTS (0): {(y == 0).sum()}")

    # Approximately equal train/validation/test split, stratified by class.
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=1 / 3,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    print(f"[INFO] Train size: {X_train.shape[0]}")
    print(f"[INFO] Val size:   {X_val.shape[0]}")
    print(f"[INFO] Test size:  {X_test.shape[0]}")

    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"],
        "max_iter": [1000, 3000],
        "class_weight": ["balanced", None],
    }

    best_params = None
    best_val_auc = -np.inf
    tuning_records = []

    print("[INFO] Tuning Logistic Regression on validation set...")

    for params in ParameterGrid(param_grid):
        pipeline = build_pipeline(
            C=params["C"],
            solver=params["solver"],
            max_iter=params["max_iter"],
            class_weight=params["class_weight"],
        )

        try:
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter("always", ConvergenceWarning)
                pipeline.fit(X_train, y_train)

            val_prob = pipeline.predict_proba(X_val)[:, 1]
            val_pred = (val_prob >= 0.5).astype(int)

            val_auc = roc_auc_score(y_val, val_prob)
            val_acc = accuracy_score(y_val, val_pred)
            val_f1 = f1_score(y_val, val_pred, zero_division=0)
            val_precision = precision_score(y_val, val_pred, zero_division=0)
            val_recall = recall_score(y_val, val_pred, zero_division=0)

            has_convergence_warning = any(
                issubclass(w.category, ConvergenceWarning)
                for w in caught_warnings
            )

            tuning_records.append(
                {
                    "C": params["C"],
                    "solver": params["solver"],
                    "max_iter": params["max_iter"],
                    "class_weight": params["class_weight"],
                    "val_auc": val_auc,
                    "val_acc": val_acc,
                    "val_f1": val_f1,
                    "val_precision": val_precision,
                    "val_recall": val_recall,
                    "convergence_warning": has_convergence_warning,
                }
            )

            print(
                f"[INFO] C={params['C']:<5} "
                f"solver={params['solver']:<10} "
                f"max_iter={params['max_iter']:<5} "
                f"class_weight={str(params['class_weight']):<8} "
                f"Val AUC={val_auc:.4f} "
                f"Val ACC={val_acc:.4f} "
                f"Val F1={val_f1:.4f}"
            )

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_params = params.copy()

        except Exception as exc:
            print(f"[WARNING] 参数组合失败: {params}, error={exc}")

    if best_params is None:
        raise RuntimeError("所有参数组合都失败了，请检查数据或参数设置。")

    tuning_df = pd.DataFrame(tuning_records).sort_values(
        "val_auc", ascending=False
    )
    tuning_df.to_csv(out_dir / "validation_tuning_results.csv", index=False)

    print(f"[INFO] Best parameters: {best_params}")
    print(f"[INFO] Best validation AUC: {best_val_auc:.4f}")

    # Refit the final model using the combined training and validation sets.
    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = pd.concat([y_train, y_val], axis=0)

    final_pipeline = build_pipeline(
        C=best_params["C"],
        solver=best_params["solver"],
        max_iter=best_params["max_iter"],
        class_weight=best_params["class_weight"],
    )
    final_pipeline.fit(X_trainval, y_trainval)

    final_scaler = final_pipeline.named_steps["scaler"]
    final_model = final_pipeline.named_steps["lr"]

    coef = final_model.coef_.ravel()
    coef_df = pd.DataFrame(
        {
            "gene": feature_cols,
            "coefficient": coef,
            "abs_coefficient": np.abs(coef),
            "coefficient_direction": np.where(
                coef > 0,
                "HTS-associated",
                np.where(coef < 0, "LTS-associated", "neutral"),
            ),
        }
    ).sort_values("abs_coefficient", ascending=False)
    coef_df.to_csv(out_dir / "lr_coefficients_final_model.csv", index=False)

    # Evaluate the final model once on the independent test set.
    print("[INFO] Evaluating final model on test set...")
    test_prob = final_pipeline.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, test_prob)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_ap = average_precision_score(y_test, test_prob)

    metrics_df = pd.DataFrame(
        [
            {
                "best_C": best_params["C"],
                "best_solver": best_params["solver"],
                "best_max_iter": best_params["max_iter"],
                "best_class_weight": best_params["class_weight"],
                "best_val_auc": best_val_auc,
                "test_auc": test_auc,
                "test_accuracy": test_acc,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_average_precision": test_ap,
            }
        ]
    )
    metrics_df.to_csv(out_dir / "test_metrics.csv", index=False)

    report = classification_report(
        y_test,
        test_pred,
        target_names=["LTS", "HTS"],
        digits=4,
        zero_division=0,
    )
    with open(
        out_dir / "classification_report.txt", "w", encoding="utf-8"
    ) as handle:
        handle.write(report)

    test_pred_df = X_test.copy()
    test_pred_df["true_label"] = y_test.values
    test_pred_df["true_class"] = y_test.map({0: "LTS", 1: "HTS"}).values
    test_pred_df["pred_label"] = test_pred
    test_pred_df["pred_class"] = (
        pd.Series(test_pred, index=X_test.index)
        .map({0: "LTS", 1: "HTS"})
        .values
    )
    test_pred_df["pred_prob_HTS"] = test_prob
    test_pred_df.to_csv(out_dir / "test_predictions.csv", index=False)

    cm = confusion_matrix(y_test, test_pred)
    save_confusion_matrix(
        cm=cm,
        class_names=["LTS", "HTS"],
        save_path=fig_dir / "test_confusion_matrix.png",
        title="Test Confusion Matrix",
    )

    plot_roc_curve(y_test, test_prob, fig_dir / "test_roc_curve.png")
    plot_pr_curve(y_test, test_prob, fig_dir / "test_pr_curve.png")
    plot_test_score_distribution(
        y_test,
        test_prob,
        fig_dir / "test_score_distribution.png",
    )

    # Explain the refitted final model. The background is the data used to fit
    # the final model (train + validation), while SHAP values are summarized
    # across the complete dataset for post hoc feature interpretation.
    print("[INFO] Calculating SHAP values for the refitted final model...")
    try:
        X_trainval_scaled = final_scaler.transform(X_trainval)
        X_all_scaled = final_scaler.transform(X)

        shap_values = get_linear_shap_values(
            model=final_model,
            background=X_trainval_scaled,
            data=X_all_scaled,
        )

        save_shap_summary_plot(
            shap_values,
            X_all_scaled,
            feature_cols,
            fig_dir / "shap_summary_beeswarm_final_model_all_data_lr.png",
        )
        save_shap_summary_plot(
            shap_values,
            X_all_scaled,
            feature_cols,
            fig_dir / "shap_summary_beeswarm_final_model_all_data_lr.svg",
        )

        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        mean_shap = shap_values.mean(axis=0)

        shap_importance_df = pd.DataFrame(
            {
                "gene": feature_cols,
                "mean_abs_shap": mean_abs_shap,
                "mean_shap": mean_shap,
                "coefficient": coef,
                "coefficient_direction": np.where(
                    coef > 0,
                    "HTS-associated",
                    np.where(coef < 0, "LTS-associated", "neutral"),
                ),
            }
        ).sort_values("mean_abs_shap", ascending=False)

        shap_importance_df.to_csv(
            out_dir / "shap_gene_importance_final_model_all_data_lr.csv",
            index=False,
        )

        shap_importance_df.head(TOP_N_SHAP).to_csv(
            out_dir / f"top{TOP_N_SHAP}_shap_genes_by_abs_final_model.csv",
            index=False,
        )

        plot_topn_shap_importance(
            shap_importance_df,
            fig_dir
            / f"top{TOP_N_SHAP}_shap_gene_importance_final_model_lr.png",
            top_n=TOP_N_SHAP,
        )

        hts_associated_df = shap_importance_df[
            shap_importance_df["coefficient"] > 0
        ].sort_values("mean_abs_shap", ascending=False)

        hts_associated_df.head(TOP_N_SHAP).to_csv(
            out_dir
            / f"top{TOP_N_SHAP}_hts_associated_genes_by_shap_final_model.csv",
            index=False,
        )

        plot_hts_associated_shap_bar(
            shap_importance_df=shap_importance_df,
            top_n=TOP_N_SHAP,
            save_path=fig_dir
            / f"top{TOP_N_SHAP}_hts_associated_genes_bar_final_model_lr.png",
        )

    except Exception as exc:
        print(f"[WARNING] SHAP 处理失败：{exc}")

    print("\n[RESULT] Done.")
    print(f"[RESULT] Output directory: {out_dir}")
    print(f"[RESULT] Best parameters: {best_params}")
    print(f"[RESULT] Best validation AUC: {best_val_auc:.4f}")
    print(f"[RESULT] Test AUC: {test_auc:.4f}")
    print(f"[RESULT] Test ACC: {test_acc:.4f}")
    print(f"[RESULT] Test F1 : {test_f1:.4f}")
    print(f"[RESULT] Test Precision: {test_precision:.4f}")
    print(f"[RESULT] Test Recall   : {test_recall:.4f}")
    print(f"[RESULT] Test AP       : {test_ap:.4f}")


if __name__ == "__main__":
    main()


[INFO] Working directory: /home/liusai/lab/new/1/machine/lr
[INFO] Reading file: /home/liusai/lab/new/1/machine/lr/df_expr.csv
[INFO] Data shape: (7646, 246)
[INFO] Feature matrix shape: (7646, 244)
[INFO] HIC (1): 3823, LIC (0): 3823
[INFO] Train size: 2548
[INFO] Val size:   2549
[INFO] Test size:  2549
[INFO] Tuning Logistic Regression on validation set...
[INFO] C=0.01  solver=liblinear  max_iter=1000  class_weight=balanced Val AUC=0.9797 Val ACC=0.9247 Val F1=0.9223
[INFO] C=0.01  solver=lbfgs      max_iter=1000  class_weight=balanced Val AUC=0.9796 Val ACC=0.9239 Val F1=0.9218
[INFO] C=0.01  solver=liblinear  max_iter=3000  class_weight=balanced Val AUC=0.9797 Val ACC=0.9247 Val F1=0.9223
[INFO] C=0.01  solver=lbfgs      max_iter=3000  class_weight=balanced Val AUC=0.9796 Val ACC=0.9239 Val F1=0.9218
[INFO] C=0.01  solver=liblinear  max_iter=1000  class_weight=None     Val AUC=0.9797 Val ACC=0.9247 Val F1=0.9223
[INFO] C=0.01  solver=lbfgs      max_iter=1000  class_weight=None   